## Действия с тензорами

Импортируем библиотеку PyTorch и проверяем её установленную версию.


In [1]:
import torch
torch.__version__

'2.12.0+cpu'

Создаём тензоры разных размерностей: 0D, 1D и 2D.


In [2]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])

Выводим содержимое двумерного тензора.


In [3]:
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

Смотрим тип данных (dtype) элементов тензора.


In [4]:
tensor2d.dtype

torch.int64

Преобразуем тензор к типу float32 и выводим результат.


In [5]:
floatvec = tensor2d.to(torch.float32)
floatvec

tensor([[1., 2., 3.],
        [4., 5., 6.]])

Получаем форму (shape) двумерного тензора.


In [6]:
tensor2d.shape

torch.Size([2, 3])

Меняем форму тензора с помощью view без изменения данных.


In [7]:
tensor2d.view(3, 2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

Транспонируем двумерный тензор, меняя местами строки и столбцы.


In [8]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

Разные способы умножения матриц с т.з. синтаксиса


In [9]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [10]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## Логистрическая регрессия

### Как работает логистическая регрессия

В этом примере показана самая простая логистическая регрессия с одним входным признаком. Такая модель нужна для бинарной классификации, когда мы хотим получить ответ в духе `0` или `1`, `нет` или `да`.

**Что означают переменные:**
- `x1` - входной признак, по которому модель делает предсказание;
- `w1` - вес, который показывает силу влияния признака на результат;
- `b` - смещение (`bias`), которое помогает двигать границу решения;
- `y` - правильная целевая метка, с которой сравнивается ответ модели.

**Как идет вычисление:**
- сначала считается линейная часть: `z = x1 * w1 + b`;
- затем к `z` применяется `sigmoid`, и получается `a` - число от `0` до `1`;
- это значение можно понимать как вероятность положительного класса;
- после этого `binary_cross_entropy(a, y)` считает ошибку между предсказанием и правильным ответом.

**Как работает `sigmoid`:**

`sigmoid(z) = 1 / (1 + e^(-z))`

Эта функция переводит любое число в диапазон от `0` до `1`. Если `z` большое и положительное, результат будет ближе к `1`. Если `z` отрицательное, результат будет ближе к `0`. Поэтому `sigmoid` удобно использовать там, где модель должна выдать вероятность.

**Как работает `binary_cross_entropy`:**

Эта функция потерь сравнивает вероятность `a` с правильной меткой `y`. Если модель предсказала верно и уверенно, ошибка будет маленькой. Если модель ошиблась, значение функции потерь вырастет. Обучение как раз и нужно для того, чтобы уменьшать эту ошибку.

**Зачем здесь градиенты:**
- `w1` и `b` созданы с `requires_grad=True`, поэтому PyTorch отслеживает все вычисления с ними;
- после вызова `loss.backward()` библиотека автоматически считает производные ошибки по этим параметрам;
- найденные градиенты попадают в `w1.grad` и `b.grad`.

**Схема вычислений:**

```text
x1
  -> z = x1 * w1 + b
  -> sigmoid(z)
  -> a = predicted probability
  -> binary_cross_entropy(a, y)
  -> loss
```

То есть логистическая регрессия сначала считает линейную комбинацию признака, потом превращает ее в вероятность и сравнивает эту вероятность с правильным ответом.

In [11]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0]) # целевая метка
x1 = torch.tensor([1.1]) # входной признак
w1 = torch.tensor([2.2], requires_grad=True) # весовой параметр
b = torch.tensor([0.0], requires_grad=True) # единица смещения bias

z = x1 * w1 + b # вход сети
a = torch.sigmoid(z) # функция активация примененная ко входу (выход)

loss = F.binary_cross_entropy(a, y) # активация и выход

# grad_L_w1 = grad(loss, w1, retain_graph=True)
# grad_L_b = grad(loss, b, retain_graph=True)

# print(grad_L_w1)
# print(grad_L_b)

In [12]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## Реализация многослойных нейронных сетей

### Как работает класс `NewralNetwork`

Этот класс наследуется от `torch.nn.Module`, поэтому PyTorch воспринимает его как полноценную нейронную сеть. Внутри класса есть две главные части: `__init__` описывает, из каких слоев состоит сеть, а `forward` задает путь, по которому входные данные проходят через эти слои.

**Что происходит в `self.layers`:**
- `Linear(num_inputs, 30)` преобразует входной вектор в 30 признаков первого скрытого слоя.
- `ReLU()` добавляет нелинейность, чтобы сеть могла учить более сложные зависимости, а не только линейные преобразования.
- `Linear(30, 20)` строит второй скрытый слой из 20 нейронов.
- `ReLU()` снова оставляет положительные значения и обнуляет отрицательные.
- `Linear(20, num_outputs)` формирует итоговый выход модели.

**Как работает `ReLU`:**

`ReLU(x) = max(0, x)`

Если на вход приходит отрицательное число, функция возвращает `0`. Если число положительное, оно проходит дальше без изменений. Благодаря этому сеть становится нелинейной: без `ReLU` несколько слоев `Linear` подряд свелись бы почти к одному линейному преобразованию.

**Как работает `forward`:**
- когда мы пишем `model(x)`, PyTorch автоматически вызывает `forward(x)`;
- `x` проходит через всю цепочку слоев, записанную в `self.layers`;
- на выходе получается `logits` - сырое предсказание модели до применения `softmax` или другой функции активации на выходе;
- затем `forward` возвращает это значение наружу.

Именно поэтому блок

```python
def forward(self, x):
    logits = self.layers(x)
    return logits
```

означает: взять входной тензор `x`, пропустить его через все слои сети и вернуть результат.

**Схема архитектуры сети:**

```text
x (num_inputs)
    -> Linear(num_inputs, 30)
    -> ReLU
    -> Linear(30, 20)
    -> ReLU
    -> Linear(20, num_outputs)
    -> logits
```

То есть сеть берет входные признаки, постепенно преобразует их в более удобное внутреннее представление и в конце выдает итоговые значения для предсказания.

In [17]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs): # входы и выходы
        super().__init__()
        self.layers = torch.nn.Sequential(
            # первый скрытый слой
            torch.nn.Linear(num_inputs, 30),
            # фукция активации между скрытыми слоями
            torch.nn.ReLU(),
            # кол-во входных словев = кол-во выходных слоев предыдущего слоя
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # слой выходных данных
            torch.nn.Linear(20, num_outputs),
        )
    def forward(self, x):
        logits = self.layers(x)
        return logits

In [19]:
model = NeuralNetwork(50, 3)
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

Эта строка считает общее число обучаемых параметров модели. `p.numel()` считает, сколько чисел хранится в каждом весе или `bias`, а `if p.requires_grad` оставляет только параметры, которые будут обучаться.

In [20]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of training parameters", num_params)

Total number of training parameters 2213


In [23]:
model.layers[0].weight

Parameter containing:
tensor([[ 0.1230, -0.0760,  0.0132,  ...,  0.0489,  0.0226, -0.0944],
        [-0.1261,  0.0340,  0.1097,  ..., -0.0650,  0.0675, -0.1125],
        [-0.0579,  0.0727,  0.1218,  ..., -0.0457, -0.1069,  0.0783],
        ...,
        [ 0.0996,  0.0594,  0.1278,  ...,  0.0365, -0.0698,  0.1140],
        [-0.0829,  0.0980,  0.1248,  ..., -0.0941, -0.0839,  0.0004],
        [-0.0004, -0.0986, -0.0877,  ...,  0.1120,  0.0373, -0.0772]],
       requires_grad=True)

In [24]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [25]:
model.layers[0].bias

Parameter containing:
tensor([-0.0211, -0.0662,  0.1369, -0.0547,  0.0421, -0.1051,  0.0909, -0.1215,
        -0.0210,  0.0630, -0.0680,  0.1392, -0.0530,  0.0495, -0.1043, -0.0789,
        -0.0666,  0.1208, -0.0678,  0.1126,  0.1369, -0.0462, -0.1413,  0.0902,
         0.0917,  0.1316, -0.0669,  0.0290, -0.1319,  0.0470],
       requires_grad=True)

In [31]:
torch.manual_seed(123) # делаем начальные веса воспроизводимыми
model = NeuralNetwork(50, 3)
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [32]:
x = torch.rand((1, 50))
x.shape

torch.Size([1, 50])

In [33]:
x

tensor([[0.2391, 0.3194, 0.8111, 0.7507, 0.3306, 0.5374, 0.2845, 0.8459, 0.2232,
         0.2083, 0.8169, 0.1084, 0.3285, 0.7185, 0.3624, 0.3084, 0.8893, 0.4179,
         0.9741, 0.3697, 0.2397, 0.8936, 0.1443, 0.1365, 0.7625, 0.1632, 0.6641,
         0.1525, 0.9830, 0.5936, 0.9120, 0.0146, 0.6323, 0.4743, 0.7467, 0.3545,
         0.9994, 0.9815, 0.7399, 0.2057, 0.8742, 0.0138, 0.7676, 0.7481, 0.7570,
         0.6432, 0.9111, 0.2246, 0.8668, 0.6961]])

Прямой проход - это вычисление выходного тензора из входного. Запись `grad_fn=<AddmmBackward0>` означает, что этот тензор получился не просто как набор чисел, а как результат операции, для которой PyTorch запомнил правило обратного прохода. `Addmm` - по имени это исторически называется addmm, потому что это “add matrix-matrix multiply”, а не порядок чтения слева направо или справа налево: по смыслу это `x @ W.T + b`, то есть сначала матричное умножение, потом прибавление смещения.

Поэтому у `out` есть ссылка на узел графа вычислений. Когда позже вызовется `loss.backward()`, PyTorch пройдет по этому графу назад и посчитает градиенты для весов, bias и, если нужно, для входа. Если тензор не участвует в вычислениях с отслеживанием градиентов, у него будет `grad_fn=None`.

In [34]:
out = model(x)
out

tensor([[-0.1670,  0.1001, -0.1219]], grad_fn=<AddmmBackward0>)